# Lupa: agente analista de datos abiertos de Colombia

**Proyecto final de Inteligencia Artificial** · Línea 4: Agentes inteligentes con tools y skills
**Equipo:** Adrián Campos, David Celorio y Martín Murillo · Universidad Santiago de Cali

Lupa responde preguntas en español sobre los datos de [datos.gov.co](https://www.datos.gov.co): encuentra el conjunto de datos, lo consulta, lo analiza con código, hace gráficas, busca contexto en la web y entrega un informe con fuentes.

| Sección | Qué hace | ¿Gasta cuota de Gemini? |
| --- | --- | --- |
| 1. Instalación | Descarga el proyecto e instala dependencias | No |
| 2. Configuración | Revisa las claves y elige el modelo | No |
| 3. Verificar el LLM | Comprueba que Gemini responde y sabe pedir una tool | Poco (4 llamadas) |
| 4. Prueba directa de las tools | Prueba cada tool sola, sin el LLM | No |
| 5. Demo del agente | Preguntas de ejemplo, con la traza paso a paso | Sí |
| 6. Interfaz web | Chat con gráficas y traza (Gradio) | Sí |
| 7. Evaluación | Banco de 12 tareas, con y sin skills, por partes | Mucho |

### Antes de empezar: claves en Secretos
En el panel izquierdo de Colab abre **Secretos** (icono de llave 🔑) y crea:

| Nombre | Obligatoria | Dónde se saca |
| --- | --- | --- |
| `GEMINI_API_KEY` | Sí | https://aistudio.google.com/apikey |
| `TAVILY_API_KEY` | Para la búsqueda web | https://app.tavily.com |
| `GROQ_API_KEY` | No (respaldo) | https://console.groq.com/keys |

Activa el interruptor **"Acceso del notebook"** en cada una. Las claves nunca se escriben en el código.

## 1. Instalación

In [ ]:
# Descarga el proyecto desde GitHub e instala sus dependencias.
# Si el repositorio no se puede clonar, la celda pide subir el .zip del proyecto.
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

REPO = "https://github.com/Scarlxdr-X/lupa-agente.git"
CARPETA = Path("/content/lupa-agente")

if not (CARPETA / "pyproject.toml").exists():
    clon = subprocess.run(["git", "clone", "-q", REPO, str(CARPETA)], capture_output=True, text=True)
    if clon.returncode != 0:
        print("No se pudo clonar el repositorio:", clon.stderr.strip()[:200])
        print("Sube el archivo lupa-agente.zip:")
        from google.colab import files
        subido = files.upload()
        nombre_zip = next(iter(subido))
        with zipfile.ZipFile(nombre_zip) as z:
            z.extractall("/content/_lupa_zip")
        raiz = next(p.parent for p in Path("/content/_lupa_zip").rglob("pyproject.toml"))
        shutil.move(str(raiz), str(CARPETA))

os.chdir(CARPETA)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Colab no recarga las rutas de Python después de `pip install -e`: agregamos src/ a mano
if str(CARPETA / "src") not in sys.path:
    sys.path.insert(0, str(CARPETA / "src"))
print("Lupa instalado en", CARPETA)

## 2. Configuración
`MODELO` es el modelo principal. Si está saturado (**error 503**) o sin cuota (**error 429**), Lupa pasa solo al siguiente de `RESPALDO`, sin perder la conversación, y lo avisa en la traza. Los nombres válidos son los que lista la sección 3.

In [ ]:
import os

MODELO = "gemini-3.8-flash"   # modelo principal de Gemini
RESPALDO = "gemini-3.5-flash,gemini-3.7-flash,gemini-3.5-flash-lite"   # se prueban en orden si el principal falla
os.environ["LUPA_MODELO_GEMINI"] = MODELO
os.environ["LUPA_MODELOS_RESPALDO_GEMINI"] = RESPALDO
os.environ["LUPA_PROVEEDOR"] = "gemini"
os.environ["LUPA_CACHE_HORAS"] = "168"   # reutiliza las consultas a datos.gov.co durante 7 días

from lupa.config import Config

config = Config.cargar()
print("Configuración de Lupa")
for clave, valor in config.resumen().items():
    print(f"  {clave:<24} {valor}")

if not config.gemini_api_key:
    print("\nFALTA GEMINI_API_KEY: créala en Secretos (🔑) y activa 'Acceso del notebook'.")
if not config.tavily_api_key:
    print("\nAviso: sin TAVILY_API_KEY el agente funciona, pero no puede buscar en la web.")

## 3. Verificar el LLM
Hace dos pruebas: que Gemini responda texto y que sepa **pedir una tool, recibir el resultado y usarlo**, que es justo lo que hace el ciclo del agente. También lista los modelos que puede usar tu clave.

In [ ]:
from lupa.verificacion import main as verificar

verificar(["--proveedor", "gemini", "--listar-modelos"]);

## 4. Prueba directa de las tools (sin el LLM)
Cada tool se llama sola, por el mismo registro que usa el agente. Así se comprueba que las APIs (datos.gov.co, Tavily) y el código local funcionan **sin gastar cuota de Gemini**.

La última prueba debe salir **bloqueada a propósito**: es una consulta a una columna con datos personales (documento del proveedor en SECOP II).

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from lupa.llm.base import LlamadaTool
from lupa.skills import cargar_skills
from lupa.tools import Contexto, crear_registro

contexto_prueba = Contexto(Path("salidas/prueba_tools"))
registro = crear_registro(config, contexto_prueba, cargar_skills())

CODIGO_GRAFICA = """
df = pd.read_csv('tabla_1.csv')
df['mes'] = pd.to_datetime(df['mes']).dt.strftime('%Y-%m')
print(df.to_string(index=False))
plt.figure(figsize=(9, 4))
plt.plot(df['mes'], df['trm'], marker='o')
plt.title('TRM promedio mensual, 2025')
plt.ylabel('Pesos colombianos por dólar (COP)')
plt.xticks(rotation=45)
plt.figtext(0.01, -0.08, 'Fuente: datos.gov.co, conjunto 32sa-8pi3', fontsize=8)
plt.savefig('trm_2025.png', dpi=120, bbox_inches='tight')
"""

PRUEBAS = [
    # (qué se prueba, tool, argumentos, ¿debe fallar?)
    ("Buscar conjuntos en el catálogo", "buscar_datasets",
     {"consulta": "tasa de cambio representativa del mercado", "limite": 2}, False),
    ("Consultar la TRM agregando en el servidor", "consultar_dataset",
     {"dataset_id": "32sa-8pi3",
      "select": "date_trunc_ym(vigenciadesde) AS mes, avg(valor) AS trm",
      "where": "vigenciadesde between '2025-01-01T00:00:00' and '2025-12-31T23:59:59'",
      "group": "mes", "order": "mes"}, False),
    ("Analizar y graficar con Python", "ejecutar_python", {"codigo": CODIGO_GRAFICA}, False),
    ("Cuenta exacta con la calculadora", "calculadora", {"expresion": "(3800.5 - 4300.25) * 1000"}, False),
    ("Buscar contexto en la web", "buscar_web",
     {"consulta": "qué movió el dólar en Colombia en 2025", "max_resultados": 2}, False),
    ("Cargar un skill", "cargar_skill", {"nombre": "explorar_datos"}, False),
    ("Privacidad: columna personal bloqueada", "consultar_dataset",
     {"dataset_id": "jbjy-vk9h", "select": "documento_proveedor", "limit": 1}, True),
]

resumen_pruebas = []
for i, (descripcion, tool, argumentos, debe_fallar) in enumerate(PRUEBAS, 1):
    resultado = registro.ejecutar(LlamadaTool(id=f"prueba-{i}", nombre=tool, argumentos=argumentos))
    paso = (not resultado.ok) if debe_fallar else resultado.ok
    resumen_pruebas.append((descripcion, tool, paso, resultado.duracion_s))
    print("=" * 100)
    print(f"{i}. {descripcion} → {tool}   [{'OK' if paso else 'FALLA'}] ({resultado.duracion_s:.1f} s)")
    print("-" * 100)
    texto = resultado.texto
    print(texto if len(texto) <= 900 else texto[:900] + "\n[...]")

for grafica in contexto_prueba.graficas:
    display(Image(filename=str(grafica)))

print("\nRESUMEN")
for descripcion, tool, paso, segundos in resumen_pruebas:
    print(f"  [{'OK   ' if paso else 'FALLA'}] {tool:<18} {descripcion} ({segundos:.1f} s)")

## 5. Demo del agente
Cada línea de la traza es un paso del ciclo **razonar → actuar → observar**: qué tool pidió el LLM, con qué argumentos y qué obtuvo. Al final se muestran la respuesta, las gráficas y las métricas.

In [ ]:
from IPython.display import Image, Markdown, display

from lupa.agent import Agente
from lupa.llm import crear_proveedor

agente = Agente(crear_proveedor(config), config, usar_skills=True)


def preguntar(pregunta):
    """Hace una pregunta al agente y muestra la traza, la respuesta, las gráficas y las métricas."""
    print(f"PREGUNTA: {pregunta}\n")
    for e in agente.responder_por_pasos(pregunta):
        if e.tipo == "pensamiento":
            print(f"[paso {e.paso}] razonamiento: {e.texto[:200]}")
        elif e.tipo == "llamada":
            print(f"[paso {e.paso}] -> {e.tool} {str(e.argumentos)[:200]}")
        elif e.tipo == "resultado":
            print(("     ok    " if e.ok else "     ERROR ") + e.texto[:160].replace("\n", " "))
        elif e.tipo in ("aviso", "error"):
            print(f"[{e.tipo}] {e.texto}")
    res = agente.ultimo_resultado
    display(Markdown(res.respuesta))
    for g in res.graficas:
        display(Image(filename=str(g)))
    print(f"\nPasos: {res.pasos} · Llamadas al LLM: {res.llamadas_llm} · "
          f"Tokens: {res.tokens_entrada + res.tokens_salida:,} · Tiempo: {res.duracion_s:.0f} s · "
          f"Skills: {', '.join(res.skills_cargados) or 'ninguno'}")
    return res

### 5.1 Consulta puntual y cálculo exacto (TRM + calculadora)

In [ ]:
preguntar("Si compré 1.000 dólares el 2 de enero de 2025 y los vendí el 30 de diciembre de 2025, "
          "¿cuánto gané o perdí en pesos?");

### 5.2 Memoria de la conversación
La pregunta no repite las fechas: el agente debe recordarlas de la anterior.

In [ ]:
preguntar("¿Y si hubieran sido 5.000 dólares?");

### 5.3 Tarea compleja: datos + gráfica + web + informe

In [ ]:
agente.reiniciar()   # empieza una conversación nueva
preguntar("¿Cuánto contrató la Gobernación del Valle del Cauca por SECOP II en 2025, por dependencia? "
          "Muestra una gráfica de las 5 dependencias con mayor valor y busca si hubo noticias sobre esa contratación.");

## 6. Interfaz web
Abre el enlace público (`https://….gradio.live`) que aparece abajo. A la izquierda está el chat; a la derecha, las gráficas, la traza del agente y las métricas.
La celda se queda ejecutándose mientras la interfaz esté abierta: para seguir con el notebook, detenla con el botón ⏹.

In [ ]:
from lupa.ui.app import construir_app

construir_app(config).queue().launch(share=True, allowed_paths=[str(CARPETA / "salidas")], debug=True)

## 7. Evaluación con y sin skills
Corre las 12 tareas de `eval/tareas.json` dos veces (**sin skills** y **con skills**) y compara la tasa de éxito, el uso de tools, los pasos, los errores recuperados, el tiempo y los tokens. Las respuestas esperadas se calculan en el momento con una consulta directa a datos.gov.co.

La evaluación completa hace entre 150 y 250 llamadas al LLM, más de lo que suele permitir la capa gratuita en un día. Por eso está **dividida en 3 partes**: corre una parte, y si aparece el error 429 (cuota agotada), sigue con la siguiente otro día. La cuota diaria se reinicia a la medianoche hora del Pacífico (3:00 a. m. en Colombia).

Los resultados se borran cuando Colab se reinicia: **después de cada parte corre la celda 7.5** para descargarlos.

### 7.1 Parte 1: tareas simples

In [ ]:
from lupa.evaluacion import main as evaluar

evaluar(["--tareas", "S1,S2,S3,S4", "--pausa", "8"]);

### 7.2 Parte 2: tareas de varios pasos

In [ ]:
evaluar(["--tareas", "M1,M2,M3,M4", "--pausa", "8"]);

### 7.3 Parte 3: tareas complejas

In [ ]:
evaluar(["--tareas", "C1,C2,C3,C4", "--pausa", "8"]);

### 7.4 Juntar las partes y ver el resultado
Une todas las corridas que haya en `eval/resultados/`. Si subiste corridas de otro día (celda 7.6), también entran.

In [ ]:
from pathlib import Path

from IPython.display import Image, Markdown, display

carpeta_resultados = CARPETA / "eval" / "resultados"
carpeta_resultados.mkdir(parents=True, exist_ok=True)
partes = sorted(p for p in carpeta_resultados.iterdir()
                if p.is_dir() and not p.name.startswith("combinado") and (p / "resultados.json").exists())
print("Corridas encontradas:", ", ".join(p.name for p in partes) or "ninguna")

if partes:
    evaluar(["--unir", *map(str, partes)])
    ultima = sorted(carpeta_resultados.glob("combinado-*"))[-1]
    display(Markdown((ultima / "resumen.md").read_text(encoding="utf-8")))
    display(Image(filename=str(ultima / "exito_por_nivel.png")))

### 7.5 Descargar los resultados

In [ ]:
import shutil

from google.colab import files

archivo = shutil.make_archive("/content/resultados_evaluacion", "zip", CARPETA / "eval" / "resultados")
files.download(archivo)

### 7.6 Subir resultados de otro día (opcional)
Si corriste una parte en otra sesión de Colab, sube aquí el `resultados_evaluacion.zip` que descargaste y luego vuelve a correr la celda 7.4.

In [ ]:
import zipfile

from google.colab import files

for nombre in files.upload():
    with zipfile.ZipFile(nombre) as z:
        z.extractall(CARPETA / "eval" / "resultados")
print("Listo. Ahora corre la celda 7.4.")